# Notebook 3 — Train LSTM Fall Detection
**Chạy trên Kaggle T4x2 · ~15 phút**

Dataset cần:
- `keypoints.zip` từ Notebook 2 → upload thành Kaggle Dataset tên `fall-keypoints`

In [ ]:
!pip install onnx onnxscript -q

In [ ]:
import os, json, shutil
import numpy as np
from pathlib import Path

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.metrics import classification_report, confusion_matrix

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {DEVICE}')
print(f'GPU: {torch.cuda.get_device_name(0)}')

In [ ]:
from pathlib import Path
import json

KP_DIR = Path('/kaggle/input/datasets/nguynteincong/keypoint')

with open(KP_DIR / 'split.json') as f:
    split = json.load(f)

print(f'Train sequences: {len(split["train"])}')
print(f'Val   sequences: {len(split["val"])}')

In [ ]:
# Bước 2: Dataset — cắt chuỗi 20 frame, nhãn = majority trong cửa sổ

SEQ_LEN = 20
STRIDE  = 5   # bước trượt → tăng số samples, không bị overlap quá nhiều

class FallWindowDataset(Dataset):
    def __init__(self, seq_names, kp_dir, seq_len=20, stride=5, augment=False):
        self.samples  = []
        self.augment  = augment
        
        for name in seq_names:
            kps    = np.load(kp_dir / f'{name}_kps.npy')     # (T, 34)
            labels = np.load(kp_dir / f'{name}_labels.npy')  # (T,)
            T = len(kps)
            
            # Trượt cửa sổ seq_len frame
            for start in range(0, T - seq_len + 1, stride):
                window = kps[start:start + seq_len]
                # Nhãn = 1 nếu > 50% frame trong cửa sổ là Fall
                label  = int(labels[start:start + seq_len].mean() > 0.5)
                self.samples.append((window.astype(np.float32), label))
        
        n_fall   = sum(1 for _, l in self.samples if l == 1)
        n_normal = len(self.samples) - n_fall
        print(f'  {"aug" if augment else "val":3s}: {len(self.samples):5d} samples '
              f'(fall={n_fall}, normal={n_normal})')
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        kps, label = self.samples[idx]
        
        if self.augment:
            # Flip ngang: x → 1-x (chỉ toạ độ x, index chẵn)
            if np.random.rand() < 0.5:
                kps = kps.copy()
                kps[:, 0::2] = 1.0 - kps[:, 0::2]
            # Thêm noise nhỏ
            kps = kps + np.random.randn(*kps.shape).astype(np.float32) * 0.01
        
        return torch.tensor(kps), torch.tensor(label, dtype=torch.long)


print('Tạo dataset...')
train_ds = FallWindowDataset(split['train'], KP_DIR, SEQ_LEN, STRIDE, augment=True)
val_ds   = FallWindowDataset(split['val'],   KP_DIR, SEQ_LEN, STRIDE, augment=False)

train_loader = DataLoader(train_ds, batch_size=64, shuffle=True,  num_workers=2)
val_loader   = DataLoader(val_ds,   batch_size=64, shuffle=False, num_workers=2)

In [ ]:
# Bước 3: Định nghĩa model LSTM

class FallLSTM(nn.Module):
    def __init__(self, input_size=34, hidden=64, n_layers=2, dropout=0.3):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size, hidden,
            num_layers  = n_layers,
            batch_first = True,
            dropout     = dropout if n_layers > 1 else 0.0,
        )
        self.head = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Dropout(dropout),
            nn.Linear(hidden, 2),  # 2 class: normal / fall
        )
    
    def forward(self, x):
        out, _ = self.lstm(x)         # (B, T, hidden)
        return self.head(out[:, -1])  # lấy timestep cuối


model = FallLSTM().to(DEVICE)
n_params = sum(p.numel() for p in model.parameters())
print(f'Model parameters: {n_params:,}  (~{n_params*4/1024:.0f} KB)')

In [ ]:
# Bước 4: Tính class weight để xử lý imbalance

labels_all = [l for _, l in train_ds.samples]
n_normal   = labels_all.count(0)
n_fall     = labels_all.count(1)
total      = len(labels_all)

# Weight tỉ lệ nghịch với tần suất
w_normal = total / (2 * n_normal)
w_fall   = total / (2 * n_fall)
weights  = torch.tensor([w_normal, w_fall], dtype=torch.float32).to(DEVICE)

print(f'Normal: {n_normal}  Fall: {n_fall}')
print(f'Class weights → normal={w_normal:.2f}  fall={w_fall:.2f}')

criterion = nn.CrossEntropyLoss(weight=weights)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=50)

In [ ]:
# Bước 5: Training loop

EPOCHS    = 50
best_f1   = 0.0
best_path = '/kaggle/working/fall_lstm_best.pt'

from sklearn.metrics import f1_score

def run_epoch(model, loader, train=True):
    model.train() if train else model.eval()
    total_loss = correct = total = 0
    all_preds = []
    all_labels = []
    
    ctx = torch.enable_grad() if train else torch.no_grad()
    with ctx:
        for x, y in loader:
            x, y = x.to(DEVICE), y.to(DEVICE)
            if train:
                optimizer.zero_grad()
            logits = model(x)
            loss   = criterion(logits, y)
            if train:
                loss.backward()
                optimizer.step()
            total_loss += loss.item() * len(y)
            preds = logits.argmax(1)
            correct  += (preds == y).sum().item()
            total    += len(y)
            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(y.cpu().numpy())
    
    f1 = f1_score(all_labels, all_preds, average='binary', zero_division=0)
    return total_loss / total, correct / total, f1, all_preds, all_labels


print(f'{'Epoch':>5}  {'TrainLoss':>9}  {'TrainAcc':>8}  {'ValLoss':>7}  {'ValAcc':>6}  {'F1-Fall':>7}')
print('─' * 60)

for epoch in range(1, EPOCHS + 1):
    tr_loss, tr_acc, _,       _, _ = run_epoch(model, train_loader, train=True)
    va_loss, va_acc, va_f1, preds, labels = run_epoch(model, val_loader, train=False)
    scheduler.step()
    
    flag = ''
    if va_f1 > best_f1:
        best_f1 = va_f1
        torch.save(model.state_dict(), best_path)
        flag = ' ← best'
    
    if epoch % 5 == 0 or epoch == 1:
        print(f'{epoch:>5}  {tr_loss:>9.4f}  {tr_acc:>8.3f}  '
              f'{va_loss:>7.4f}  {va_acc:>6.3f}  {va_f1:>7.4f}{flag}')

print(f'\nBest F1-Fall: {best_f1:.4f}')

In [ ]:
# Bước 6: Báo cáo chi tiết với best model

model.load_state_dict(torch.load(best_path))
_, _, _, preds, labels = run_epoch(model, val_loader, train=False)

print('Classification Report:')
print(classification_report(labels, preds, target_names=['Normal', 'Fall']))

print('Confusion Matrix:')
print(confusion_matrix(labels, preds))
print('(rows=actual, cols=predicted)')

In [ ]:
# Bước 7: Export ONNX

model.eval()
dummy    = torch.zeros(1, SEQ_LEN, 34).to(DEVICE)
onnx_path = '/kaggle/working/fall_lstm.onnx'

torch.onnx.export(
    model, dummy, onnx_path,
    input_names   = ['keypoints'],
    output_names  = ['logits'],
    dynamic_axes  = {'keypoints': {0: 'batch'}},
    opset_version = 12,
)

print(f'ONNX saved: {onnx_path}')
print(f'Size: {os.path.getsize(onnx_path) / 1024:.1f} KB')

In [ ]:
# Bước 8: Tổng kết files output

print('=== Files sẵn sàng download ===')
outputs = [
    ('/kaggle/working/fall_lstm_best.pt',  'PyTorch weights'),
    ('/kaggle/working/fall_lstm.onnx',     'ONNX → convert TensorRT trên Jetson'),
]
for path, desc in outputs:
    size = os.path.getsize(path) / 1024
    print(f'  {desc:<40} {size:6.1f} KB  {path}')

print('\n=== Bước tiếp theo trên Jetson Nano ===')
print('Có 3 file ONNX cần convert:')
print('  1. fire_best.onnx       (từ notebook 1)')
print('  2. yolov8n-pose.onnx    (export từ ultralytics)')
print('  3. fall_lstm.onnx       (file này)')
print('')
print('Lệnh convert TensorRT trên Jetson Nano:')
print('  trtexec --onnx=fire_best.onnx    --saveEngine=fire.engine    --fp16')
print('  trtexec --onnx=yolov8n-pose.onnx --saveEngine=pose.engine    --fp16')
print('  trtexec --onnx=fall_lstm.onnx    --saveEngine=lstm.engine    --fp16')